# Phase 10 — Ensemble Learning

## H&M Personalized Fashion Recommendations → Customer Purchase Prediction

### Objective

Phase 8 compared individual model families and Phase 9 studied optimization and gradient-based learning.

Phase 10 focuses on **ensemble learning**:

> Can combining multiple models produce a more accurate and robust predictor than a single model?

We will study:

- Bagging
- Random Forest
- Extra Trees
- Boosting
- Gradient Boosting
- XGBoost
- Soft Voting
- Hard Voting
- Model diversity
- Error correlation
- Stacking
- Ensemble vs individual-model performance
- Performance vs computational cost

---

## Core idea

Instead of relying on one model:

```text
Dataset
   ↓
One model
   ↓
Prediction
```

an ensemble uses multiple learners:

```text
                ┌── Model 1 ──┐
Dataset ────────┼── Model 2 ──┼──→ Aggregation → Final prediction
                ├── Model 3 ──┤
                └── Model N ──┘
```

The main reason ensembles work is **error diversity**.

If different models make different mistakes, combining them can cancel some of those errors.

> The temporal train/validation/test framework from Phase 5 is preserved. The test set is not used for ensemble construction or selection.


# 10.1 Ensemble Learning Taxonomy

## Bagging

Bagging trains multiple models independently and aggregates their predictions.

```text
Bootstrap/sample 1 → Model 1 ─┐
Bootstrap/sample 2 → Model 2 ─┤
Bootstrap/sample 3 → Model 3 ─┼→ Aggregate
Bootstrap/sample N → Model N ─┘
```

Main goal:

> **Reduce variance.**

Random Forest is a classic bagging-style ensemble.

---

## Boosting

Boosting trains models sequentially.

```text
Model 1
   ↓
Errors
   ↓
Model 2 focuses on errors
   ↓
Errors
   ↓
Model 3
   ↓
Final weighted ensemble
```

Main goal:

> **Reduce bias while controlling variance.**

Examples:

- Gradient Boosting
- XGBoost

---

## Voting

Different independently trained models vote on the final prediction.

### Hard voting

Uses predicted classes.

### Soft voting

Uses predicted probabilities.

Soft voting is usually more informative for probabilistic classification.

---

## Stacking

Stacking uses the predictions of base models as inputs to a **meta-model**.

```text
                 ┌── Logistic Regression ──┐
Dataset ─────────┼── Random Forest ────────┼→ Meta-model → Final
                 ├── Extra Trees ──────────┤
                 └── Boosting ─────────────┘
```

The meta-model learns how to combine the base models.


# 10.2 Why Ensemble Diversity Matters

Suppose two models have:

```text
Model A accuracy = 85%
Model B accuracy = 85%
```

This does not automatically mean combining them will help.

If both models make exactly the same errors:

```text
Error A ≈ Error B
```

there is little benefit from combining them.

But if:

```text
Error A ≠ Error B
```

then an ensemble may correct some mistakes.

Therefore, we will explicitly measure **prediction correlation** and disagreement.


# 10.3 Imports and Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import time
import json
import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

RANDOM_STATE = 42

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = PROCESSED_DIR / "train_phase5.parquet"
VAL_PATH = PROCESSED_DIR / "validation_phase5.parquet"
TEST_PATH = PROCESSED_DIR / "test_phase5.parquet"

PREPROCESSOR_PATH = (
    MODELS_DIR / "preprocessor_standard_phase6.joblib"
)

print("Project root:", PROJECT_ROOT.resolve())


# 10.4 Load Phase 5 Data

In [ ]:
train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)
test_df = pd.read_parquet(TEST_PATH)

TARGET_COL = "target"
ID_COL = "customer_id"

X_train_raw = train_df.drop(
    columns=[TARGET_COL, ID_COL]
)
X_val_raw = val_df.drop(
    columns=[TARGET_COL, ID_COL]
)
X_test_raw = test_df.drop(
    columns=[TARGET_COL, ID_COL]
)

y_train = train_df[TARGET_COL].astype("int8")
y_val = val_df[TARGET_COL].astype("int8")
y_test = test_df[TARGET_COL].astype("int8")

print("Train:", X_train_raw.shape)
print("Validation:", X_val_raw.shape)
print("Test:", X_test_raw.shape)

print("\nTarget rates:")
print("Train:", y_train.mean())
print("Validation:", y_val.mean())
print("Test:", y_test.mean())


# 10.5 Load Phase 6 Preprocessing

We use the already-fitted preprocessing pipeline.

This ensures the ensemble experiments use the same feature representation established earlier.

No preprocessing is fitted using validation or test data.


In [ ]:
preprocessor = joblib.load(
    PREPROCESSOR_PATH
)

X_train_processed = preprocessor.transform(
    X_train_raw
)
X_val_processed = preprocessor.transform(
    X_val_raw
)
X_test_processed = preprocessor.transform(
    X_test_raw
)

print("Processed train:", X_train_processed.shape)
print("Processed validation:", X_val_processed.shape)
print("Sparse:", sp.issparse(X_train_processed))


# 10.6 Computational Safety

Tree ensembles can become expensive on the full H&M feature matrix.

We therefore use a configurable sample for the ensemble experiments if the dense representation would require excessive memory.

The validation sample is also controlled.

The goal here is to compare **ensemble mechanisms and diversity**, while final full-data training can be performed after the best configuration has been selected.


In [ ]:
MAX_DENSE_GB = 4.0
ENSEMBLE_SAMPLE_SIZE = 200_000

def estimated_dense_gb(matrix, dtype=np.float32):
    rows, cols = matrix.shape
    return (
        rows * cols * np.dtype(dtype).itemsize
        / (1024 ** 3)
    )

train_dense_gb = estimated_dense_gb(
    X_train_processed
)
val_dense_gb = estimated_dense_gb(
    X_val_processed
)

print(
    f"Estimated train dense memory: "
    f"{train_dense_gb:.2f} GB"
)

print(
    f"Estimated validation dense memory: "
    f"{val_dense_gb:.2f} GB"
)


In [ ]:
rng = np.random.default_rng(
    RANDOM_STATE
)

if train_dense_gb <= MAX_DENSE_GB:
    X_train_ens = X_train_processed.astype(
        np.float32
    ).toarray()

    y_train_ens = y_train.to_numpy()

    train_mode = "full"
else:
    sample_size = min(
        ENSEMBLE_SAMPLE_SIZE,
        X_train_processed.shape[0]
    )

    train_indices = rng.choice(
        X_train_processed.shape[0],
        size=sample_size,
        replace=False
    )

    X_train_ens = X_train_processed[
        train_indices
    ].astype(np.float32).toarray()

    y_train_ens = y_train.iloc[
        train_indices
    ].to_numpy()

    train_mode = "sampled"

if val_dense_gb <= MAX_DENSE_GB:
    X_val_ens = X_val_processed.astype(
        np.float32
    ).toarray()

    y_val_ens = y_val.to_numpy()

    val_mode = "full"
else:
    val_sample_size = min(
        ENSEMBLE_SAMPLE_SIZE,
        X_val_processed.shape[0]
    )

    val_indices = rng.choice(
        X_val_processed.shape[0],
        size=val_sample_size,
        replace=False
    )

    X_val_ens = X_val_processed[
        val_indices
    ].astype(np.float32).toarray()

    y_val_ens = y_val.iloc[
        val_indices
    ].to_numpy()

    val_mode = "sampled"

print("Training mode:", train_mode)
print("Validation mode:", val_mode)
print("Ensemble train shape:", X_train_ens.shape)
print("Ensemble validation shape:", X_val_ens.shape)


# 10.7 Evaluation Function

In [ ]:
def evaluate_predictions(
    model_name,
    split_name,
    y_true,
    y_pred,
    y_prob,
    fit_time=None,
    prediction_time=None
):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    return {
        "model": model_name,
        "split": split_name,
        "accuracy": accuracy_score(
            y_true, y_pred
        ),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_true,
            y_prob
        ),
        "pr_auc": average_precision_score(
            y_true,
            y_prob
        ),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "fit_time_seconds": fit_time,
        "prediction_time_seconds": prediction_time
    }


def fit_evaluate_model(
    model,
    name,
    X_train,
    y_train,
    X_val,
    y_val
):
    start = time.perf_counter()

    model.fit(
        X_train,
        y_train
    )

    fit_time = time.perf_counter() - start

    start = time.perf_counter()
    train_prob = model.predict_proba(
        X_train
    )[:, 1]
    train_pred = (
        train_prob >= 0.5
    ).astype(int)
    train_pred_time = (
        time.perf_counter() - start
    )

    start = time.perf_counter()
    val_prob = model.predict_proba(
        X_val
    )[:, 1]
    val_pred = (
        val_prob >= 0.5
    ).astype(int)
    val_pred_time = (
        time.perf_counter() - start
    )

    results = pd.DataFrame([
        evaluate_predictions(
            name,
            "train",
            y_train,
            train_pred,
            train_prob,
            fit_time,
            train_pred_time
        ),
        evaluate_predictions(
            name,
            "validation",
            y_val,
            val_pred,
            val_prob,
            fit_time,
            val_pred_time
        )
    ])

    return model, results, train_prob, val_prob


# 10.8 Base Model — Logistic Regression

Logistic Regression gives us a relatively low-variance linear base learner.

It is useful inside an ensemble because it makes different assumptions from tree-based learners.


In [ ]:
logistic_base = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="liblinear",
    max_iter=1000,
    random_state=RANDOM_STATE
)

logistic_base, logistic_results, logistic_train_prob, logistic_val_prob = (
    fit_evaluate_model(
        logistic_base,
        "LogisticRegression",
        X_train_processed,
        y_train,
        X_val_processed,
        y_val
    )
)

display(logistic_results)


# 10.9 Base Model — Decision Tree

A single Decision Tree is intentionally included as a high-variance base learner.

Its errors will later be compared with Random Forest and Extra Trees.

This demonstrates the role of ensemble learning in reducing the instability of individual trees.


In [ ]:
decision_tree = DecisionTreeClassifier(
    max_depth=10,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

decision_tree, dt_results, dt_train_prob, dt_val_prob = (
    fit_evaluate_model(
        decision_tree,
        "DecisionTree",
        X_train_ens,
        y_train_ens,
        X_val_ens,
        y_val_ens
    )
)

display(dt_results)


# 10.10 Base Ensemble — Random Forest

Random Forest combines many randomized decision trees.

Important variance-reduction mechanisms include:

- bootstrap sampling,
- random feature selection,
- aggregation across many trees.

We use a controlled number of estimators for the initial comparison.


In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=150,
    max_depth=14,
    min_samples_leaf=20,
    max_features="sqrt",
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

random_forest, rf_results, rf_train_prob, rf_val_prob = (
    fit_evaluate_model(
        random_forest,
        "RandomForest",
        X_train_ens,
        y_train_ens,
        X_val_ens,
        y_val_ens
    )
)

display(rf_results)


# 10.11 Base Ensemble — Extra Trees

Extra Trees adds even more randomness to the construction of individual trees.

This gives us a useful comparison:

```text
Decision Tree
      ↓
Random Forest
      ↓
Extra Trees
```

We can then examine whether increasing randomization improves validation generalization.


In [ ]:
extra_trees = ExtraTreesClassifier(
    n_estimators=150,
    max_depth=14,
    min_samples_leaf=20,
    max_features="sqrt",
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

extra_trees, et_results, et_train_prob, et_val_prob = (
    fit_evaluate_model(
        extra_trees,
        "ExtraTrees",
        X_train_ens,
        y_train_ens,
        X_val_ens,
        y_val_ens
    )
)

display(et_results)


# 10.12 Gradient Boosting

Gradient Boosting is fundamentally different from bagging.

Bagging:

```text
models trained independently
          ↓
aggregate
```

Boosting:

```text
model 1
  ↓
focus on errors
  ↓
model 2
  ↓
focus on remaining errors
  ↓
model 3
  ↓
...
```

This sequential correction mechanism can reduce bias effectively.

For computational reasons, we use a controlled configuration.


In [ ]:
gradient_boosting = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=3,
    min_samples_leaf=50,
    subsample=0.8,
    random_state=RANDOM_STATE
)

gradient_boosting, gb_results, gb_train_prob, gb_val_prob = (
    fit_evaluate_model(
        gradient_boosting,
        "GradientBoosting",
        X_train_ens,
        y_train_ens,
        X_val_ens,
        y_val_ens
    )
)

display(gb_results)


# 10.13 XGBoost

XGBoost is included when available.

It extends the gradient-boosting idea with:

- regularization,
- efficient tree construction,
- histogram-based training,
- flexible sampling,
- strong tabular-data performance.

The configuration here is deliberately conservative because extensive tuning is reserved for a later phase.


In [ ]:
try:
    from xgboost import XGBClassifier
    xgb_available = True
    print("XGBoost is available.")
except ImportError:
    xgb_available = False
    print(
        "XGBoost is not installed. "
        "The XGBoost experiment will be skipped."
    )


In [ ]:
xgb_model = None
xgb_results = pd.DataFrame()
xgb_train_prob = None
xgb_val_prob = None

if xgb_available:

    positive_rate = float(
        np.mean(y_train_ens)
    )

    negative_rate = (
        1.0 - positive_rate
    )

    scale_pos_weight = (
        negative_rate / positive_rate
        if positive_rate > 0
        else 1.0
    )

    xgb_model = XGBClassifier(
        n_estimators=250,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=10,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=RANDOM_STATE,
        scale_pos_weight=scale_pos_weight
    )

    xgb_model, xgb_results,     xgb_train_prob, xgb_val_prob = (
        fit_evaluate_model(
            xgb_model,
            "XGBoost",
            X_train_ens,
            y_train_ens,
            X_val_ens,
            y_val_ens
        )
    )

    display(xgb_results)


# 10.14 Individual-Model Leaderboard

Before building a new ensemble, we need to understand the individual learners.

This establishes the benchmark against which voting and stacking will be compared.


In [ ]:
individual_results = pd.concat(
    [
        logistic_results,
        dt_results,
        rf_results,
        et_results,
        gb_results,
        xgb_results
    ],
    ignore_index=True
)

individual_validation = (
    individual_results[
        individual_results["split"] == "validation"
    ]
    .sort_values(
        ["pr_auc", "f1"],
        ascending=False
    )
)

display(
    individual_validation[
        [
            "model",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "fit_time_seconds"
        ]
    ]
)


# 10.15 Prediction Diversity

We now examine the validation probabilities produced by different models.

The correlation matrix answers:

> Do different models make similar predictions?

High correlation:

```text
Model A ≈ Model B
```

means they behave similarly.

Lower correlation suggests greater diversity.

A useful ensemble often combines models that are both strong **and sufficiently different**.


In [ ]:
probability_dict = {
    "LogisticRegression": logistic_val_prob,
    "DecisionTree": dt_val_prob,
    "RandomForest": rf_val_prob,
    "ExtraTrees": et_val_prob,
    "GradientBoosting": gb_val_prob
}

if xgb_val_prob is not None:
    probability_dict["XGBoost"] = xgb_val_prob

probability_df = pd.DataFrame(
    probability_dict
)

correlation_matrix = (
    probability_df.corr()
)

display(correlation_matrix)


# 10.16 Plot Prediction Correlation

This visualization makes ensemble diversity easier to inspect.

Remember:

> Low correlation is useful only if the models are individually competent.

A weak model that behaves randomly is not automatically a valuable ensemble component.


In [ ]:
fig, ax = plt.subplots(
    figsize=(10, 8)
)

image = ax.imshow(
    correlation_matrix.values,
    aspect="auto"
)

ax.set_xticks(
    range(len(correlation_matrix.columns))
)

ax.set_yticks(
    range(len(correlation_matrix.index))
)

ax.set_xticklabels(
    correlation_matrix.columns,
    rotation=45,
    ha="right"
)

ax.set_yticklabels(
    correlation_matrix.index
)

ax.set_title(
    "Validation Prediction Correlation"
)

plt.colorbar(
    image,
    ax=ax,
    label="Correlation"
)

plt.tight_layout()
plt.show()


# 10.17 Prediction Disagreement

Another diversity measure is disagreement.

For two classifiers:

\[
Disagreement
=
P(\hat y_A \neq \hat y_B)
\]

Higher disagreement means the models classify a larger fraction of observations differently.

We calculate pairwise disagreement using the default 0.50 threshold.


In [ ]:
prediction_dict = {}

for name, probability in probability_dict.items():
    prediction_dict[name] = (
        probability >= 0.5
    ).astype(int)

prediction_df = pd.DataFrame(
    prediction_dict
)

model_names = prediction_df.columns.tolist()

disagreement_matrix = pd.DataFrame(
    index=model_names,
    columns=model_names,
    dtype=float
)

for model_a in model_names:
    for model_b in model_names:

        disagreement_matrix.loc[
            model_a,
            model_b
        ] = np.mean(
            prediction_df[model_a]
            !=
            prediction_df[model_b]
        )

display(disagreement_matrix)


# 10.18 Soft Voting Ensemble

Soft voting combines predicted probabilities.

For models \(M_1, M_2, ..., M_k\):

\[
P_{ensemble}(y=1)
=
\sum_{i=1}^{k}
w_iP_i(y=1)
\]

where:

\[
\sum_i w_i = 1
\]

A simple equal-weight ensemble uses:

\[
w_i = \frac{1}{k}
\]

We start with equal-weight soft voting.


In [ ]:
soft_vote_prob = (
    probability_df.mean(axis=1)
)

soft_vote_pred = (
    soft_vote_prob >= 0.5
).astype(int)

soft_vote_result = pd.DataFrame([
    evaluate_predictions(
        "SoftVoting_EqualWeights",
        "validation",
        y_val_ens,
        soft_vote_pred,
        soft_vote_prob
    )
])

display(soft_vote_result)


# 10.19 Weighted Soft Voting

Not all models are equally strong.

We therefore test a simple performance-weighted approach.

The weights are based on validation PR-AUC.

> Important: this is acceptable only because the validation set is explicitly being used for model selection. We will not use the test set to derive weights.


In [ ]:
validation_scores = (
    individual_validation
    .set_index("model")["pr_auc"]
)

available_models = [
    name for name in probability_df.columns
    if name in validation_scores.index
]

raw_weights = np.array([
    validation_scores[name]
    for name in available_models
])

normalized_weights = (
    raw_weights / raw_weights.sum()
)

weighted_probability_df = (
    probability_df[available_models]
    .copy()
)

weighted_soft_vote_prob = (
    weighted_probability_df.to_numpy()
    @ normalized_weights
)

weighted_soft_vote_pred = (
    weighted_soft_vote_prob >= 0.5
).astype(int)

weighted_soft_vote_result = pd.DataFrame([
    evaluate_predictions(
        "SoftVoting_PR_AUC_Weighted",
        "validation",
        y_val_ens,
        weighted_soft_vote_pred,
        weighted_soft_vote_prob
    )
])

print("Weights:")
display(
    pd.DataFrame({
        "model": available_models,
        "weight": normalized_weights
    })
)

display(weighted_soft_vote_result)


# 10.20 Hard Voting

Hard voting combines predicted classes.

For binary classification:

```text
Model 1 → 0
Model 2 → 1
Model 3 → 1
Model 4 → 1
       ↓
Majority vote → 1
```

Hard voting ignores the confidence of individual models.

This is why soft voting is often more informative when calibrated probabilities are available.


In [ ]:
hard_vote_count = (
    prediction_df.sum(axis=1)
)

hard_vote_threshold = (
    len(prediction_df.columns) / 2
)

hard_vote_pred = (
    hard_vote_count >= hard_vote_threshold
).astype(int)

hard_vote_prob = (
    probability_df.mean(axis=1)
)

hard_vote_result = pd.DataFrame([
    evaluate_predictions(
        "HardVoting",
        "validation",
        y_val_ens,
        hard_vote_pred,
        hard_vote_prob
    )
])

display(hard_vote_result)


# 10.21 Stacking

Stacking is a two-level learning system.

### Level 1 — Base learners

```text
Logistic Regression
Random Forest
Extra Trees
Gradient Boosting
```

Each generates predictions.

### Level 2 — Meta learner

The meta-model learns:

```text
base predictions
      ↓
meta-model
      ↓
final prediction
```

A critical issue is **data leakage**.

If the meta-model is trained on predictions generated by base models on the same data they were trained on, the base predictions can be overly optimistic.

Scikit-learn's `StackingClassifier` uses cross-validation to generate training predictions for the meta-model, which is why it is preferable to manually stacking in-sample predictions.


# 10.22 Build Stacking Ensemble

We use a small collection of diverse base learners.

The meta-model is Logistic Regression.

The base learners are trained on the ensemble representation created above.


In [ ]:
stack_estimators = [
    (
        "dt",
        DecisionTreeClassifier(
            max_depth=8,
            min_samples_leaf=50,
            class_weight="balanced",
            random_state=RANDOM_STATE
        )
    ),
    (
        "rf",
        RandomForestClassifier(
            n_estimators=100,
            max_depth=12,
            min_samples_leaf=20,
            max_features="sqrt",
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE
        )
    ),
    (
        "et",
        ExtraTreesClassifier(
            n_estimators=100,
            max_depth=12,
            min_samples_leaf=20,
            max_features="sqrt",
            class_weight="balanced",
            n_jobs=-1,
            random_state=RANDOM_STATE
        )
    ),
    (
        "gb",
        GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.05,
            max_depth=3,
            min_samples_leaf=50,
            random_state=RANDOM_STATE
        )
    )
]

stacking_model = StackingClassifier(
    estimators=stack_estimators,
    final_estimator=LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ),
    stack_method="predict_proba",
    cv=3,
    n_jobs=-1,
    passthrough=False
)

stacking_model, stacking_results, stacking_train_prob, stacking_val_prob = (
    fit_evaluate_model(
        stacking_model,
        "Stacking",
        X_train_ens,
        y_train_ens,
        X_val_ens,
        y_val_ens
    )
)

display(stacking_results)


# 10.23 Ensemble Leaderboard

Now we compare:

- individual models,
- hard voting,
- equal-weight soft voting,
- validation-weighted soft voting,
- stacking.

The important question is not:

> Did the ensemble become more complex?

It is:

> **Did the ensemble improve future-period validation performance enough to justify its complexity?**


In [ ]:
ensemble_results = pd.concat(
    [
        individual_results,
        soft_vote_result,
        weighted_soft_vote_result,
        hard_vote_result,
        stacking_results
    ],
    ignore_index=True
)

ensemble_validation = (
    ensemble_results[
        ensemble_results["split"] == "validation"
    ]
    .sort_values(
        ["pr_auc", "f1"],
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    ensemble_validation[
        [
            "model",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "fit_time_seconds",
            "prediction_time_seconds"
        ]
    ]
)


# 10.24 Ensemble vs Best Individual Model

We quantify the improvement over the strongest individual model.

For a metric \(M\):

\[
Improvement
=
M_{ensemble}
-
M_{best\ individual}
\]

We focus on validation PR-AUC and F1.


In [ ]:
best_individual = individual_validation.iloc[0]

best_individual_name = (
    best_individual["model"]
)

best_individual_pr_auc = (
    best_individual["pr_auc"]
)

best_individual_f1 = (
    best_individual["f1"]
)

comparison_rows = []

for _, row in ensemble_validation.iterrows():

    comparison_rows.append({
        "model": row["model"],
        "pr_auc": row["pr_auc"],
        "pr_auc_improvement_vs_best_individual": (
            row["pr_auc"]
            -
            best_individual_pr_auc
        ),
        "f1": row["f1"],
        "f1_improvement_vs_best_individual": (
            row["f1"]
            -
            best_individual_f1
        )
    })

ensemble_improvement_df = pd.DataFrame(
    comparison_rows
)

print(
    "Best individual model:",
    best_individual_name
)

display(ensemble_improvement_df)


# 10.25 Visualize Ensemble Performance

In [ ]:
plot_df = ensemble_validation.copy()

fig, ax = plt.subplots(
    figsize=(13, 7)
)

ax.bar(
    plot_df["model"],
    plot_df["pr_auc"]
)

ax.set_ylabel("Validation PR-AUC")
ax.set_xlabel("Model")
ax.set_title(
    "Individual Models vs Ensemble Models"
)

plt.xticks(
    rotation=40,
    ha="right"
)

plt.tight_layout()
plt.show()


# 10.26 Generalization Gap for Ensemble Models

A complex ensemble can still overfit.

We compare:

\[
F1_{train} - F1_{validation}
\]

and:

\[
PR\text{-}AUC_{train}
-
PR\text{-}AUC_{validation}
\]

A strong ensemble should ideally improve validation performance without producing an extreme gap.


In [ ]:
train_ensemble = ensemble_results[
    ensemble_results["split"] == "train"
].set_index("model")

val_ensemble = ensemble_results[
    ensemble_results["split"] == "validation"
].set_index("model")

ensemble_gap_rows = []

for model_name in val_ensemble.index:

    if model_name not in train_ensemble.index:
        continue

    ensemble_gap_rows.append({
        "model": model_name,
        "train_f1": train_ensemble.loc[
            model_name, "f1"
        ],
        "validation_f1": val_ensemble.loc[
            model_name, "f1"
        ],
        "f1_gap": (
            train_ensemble.loc[
                model_name, "f1"
            ]
            -
            val_ensemble.loc[
                model_name, "f1"
            ]
        ),
        "train_pr_auc": train_ensemble.loc[
            model_name, "pr_auc"
        ],
        "validation_pr_auc": val_ensemble.loc[
            model_name, "pr_auc"
        ],
        "pr_auc_gap": (
            train_ensemble.loc[
                model_name, "pr_auc"
            ]
            -
            val_ensemble.loc[
                model_name, "pr_auc"
            ]
        )
    })

ensemble_gap_df = pd.DataFrame(
    ensemble_gap_rows
)

display(
    ensemble_gap_df.sort_values(
        "validation_pr_auc",
        ascending=False
    )
)


# 10.27 Bagging vs Boosting

We can now compare the two major ensemble strategies.

### Bagging

Represented by:

- Random Forest
- Extra Trees

Main objective:

> Reduce variance.

### Boosting

Represented by:

- Gradient Boosting
- XGBoost

Main objective:

> Sequentially improve weak learners and reduce bias.

The comparison is empirical: the validation results tell us which strategy works better for this particular dataset.


In [ ]:
strategy_map = {
    "RandomForest": "Bagging",
    "ExtraTrees": "Bagging",
    "GradientBoosting": "Boosting",
    "XGBoost": "Boosting",
    "SoftVoting_EqualWeights": "Voting",
    "SoftVoting_PR_AUC_Weighted": "Voting",
    "HardVoting": "Voting",
    "Stacking": "Stacking"
}

strategy_rows = []

for _, row in ensemble_validation.iterrows():

    model_name = row["model"]

    if model_name in strategy_map:

        strategy_rows.append({
            "model": model_name,
            "ensemble_strategy": strategy_map[
                model_name
            ],
            "validation_pr_auc": row["pr_auc"],
            "validation_f1": row["f1"],
            "validation_roc_auc": row["roc_auc"]
        })

strategy_df = pd.DataFrame(
    strategy_rows
)

display(strategy_df)


# 10.28 When Does an Ensemble Help?

An ensemble is especially useful when:

### Condition 1 — Base models are reasonably strong

Combining very weak models is unlikely to produce a strong system.

### Condition 2 — Base models make different errors

Diversity creates the opportunity for error cancellation.

### Condition 3 — The aggregation method is appropriate

Soft voting, hard voting and stacking make different assumptions.

### Condition 4 — Complexity is justified

If stacking improves PR-AUC by only a tiny amount while multiplying inference cost, a simpler model may be preferable.

Therefore:

> **Best predictive score ≠ automatically best production model.**


# 10.29 Select the Phase 10 Candidate

We select the provisional candidate using validation PR-AUC.

Before accepting it, we inspect:

- PR-AUC
- F1
- ROC-AUC
- train-validation gap
- prediction cost
- model complexity

The test set is not consulted.


In [ ]:
candidate_table = ensemble_validation[
    [
        "model",
        "pr_auc",
        "roc_auc",
        "f1",
        "precision",
        "recall",
        "fit_time_seconds",
        "prediction_time_seconds"
    ]
].copy()

candidate_table = candidate_table.merge(
    ensemble_gap_df[
        [
            "model",
            "f1_gap",
            "pr_auc_gap"
        ]
    ],
    on="model",
    how="left"
)

display(candidate_table)


In [ ]:
phase10_candidate = (
    ensemble_validation.iloc[0]["model"]
)

print(
    "Provisional Phase 10 candidate:",
    phase10_candidate
)

candidate_row = ensemble_validation.iloc[0]

print(
    "Validation PR-AUC:",
    candidate_row["pr_auc"]
)

print(
    "Validation F1:",
    candidate_row["f1"]
)


# 10.30 Save Phase 10 Artifacts

We save:

- all model results,
- ensemble leaderboard,
- prediction correlations,
- disagreement matrix,
- generalization gaps,
- strategy comparison,
- experiment configuration.

Models that were actually trained are also saved.


In [ ]:
ensemble_results.to_csv(
    RESULTS_DIR / "phase10_ensemble_results.csv",
    index=False
)

ensemble_validation.to_csv(
    RESULTS_DIR / "phase10_ensemble_leaderboard.csv",
    index=False
)

correlation_matrix.to_csv(
    RESULTS_DIR / "phase10_prediction_correlation.csv"
)

disagreement_matrix.to_csv(
    RESULTS_DIR / "phase10_prediction_disagreement.csv"
)

ensemble_gap_df.to_csv(
    RESULTS_DIR / "phase10_generalization_gaps.csv",
    index=False
)

strategy_df.to_csv(
    RESULTS_DIR / "phase10_strategy_comparison.csv",
    index=False
)

ensemble_improvement_df.to_csv(
    RESULTS_DIR / "phase10_ensemble_improvement.csv",
    index=False
)

joblib.dump(
    random_forest,
    MODELS_DIR / "random_forest_phase10.joblib"
)

joblib.dump(
    extra_trees,
    MODELS_DIR / "extra_trees_phase10.joblib"
)

joblib.dump(
    gradient_boosting,
    MODELS_DIR / "gradient_boosting_phase10.joblib"
)

joblib.dump(
    stacking_model,
    MODELS_DIR / "stacking_phase10.joblib"
)

if xgb_model is not None:
    joblib.dump(
        xgb_model,
        MODELS_DIR / "xgboost_phase10.joblib"
    )

print("Phase 10 model and result artifacts saved.")


# 10.31 Save Phase 10 Configuration

In [ ]:
phase10_config = {
    "random_state": RANDOM_STATE,
    "max_dense_gb": MAX_DENSE_GB,
    "ensemble_sample_size": ENSEMBLE_SAMPLE_SIZE,
    "train_mode": train_mode,
    "validation_mode": val_mode,
    "xgboost_available": xgb_available,
    "voting_methods": [
        "hard voting",
        "equal-weight soft voting",
        "validation PR-AUC weighted soft voting"
    ],
    "stacking": {
        "meta_model": "LogisticRegression",
        "cv": 3,
        "stack_method": "predict_proba"
    },
    "test_set_used_for_selection": False,
    "phase10_candidate": phase10_candidate
}

with open(
    RESULTS_DIR / "phase10_experiment_config.json",
    "w"
) as f:
    json.dump(
        phase10_config,
        f,
        indent=4
    )

print("Configuration saved.")


# 10.32 Final Interpretation Checklist

After running the notebook, answer these questions.

## Bagging

1. Does Random Forest outperform a single Decision Tree?
2. Does Extra Trees outperform Random Forest?
3. Did ensemble averaging reduce the train-validation gap?

## Boosting

4. Does Gradient Boosting outperform the bagging ensembles?
5. Does XGBoost improve further?
6. Is the improvement large enough to justify its computational cost?

## Voting

7. Does soft voting outperform the best individual model?
8. Does weighted soft voting improve over equal weighting?
9. Does hard voting lose information compared with soft voting?

## Diversity

10. Which models have highly correlated predictions?
11. Which models have the greatest disagreement?
12. Does greater diversity actually translate into better ensemble performance?

## Stacking

13. Does stacking improve validation PR-AUC?
14. Does stacking introduce substantially more training/inference cost?
15. Is the improvement worth the additional complexity?

## Model-selection principle

The final decision should be based on:

```text
Predictive performance
        +
Generalization
        +
Robustness
        +
Computational cost
        +
Interpretability
        ↓
Final model choice
```

not on one metric alone.


# 10.33 Important Leakage Rule

Do not use the test set for:

- ensemble weights,
- selecting base learners,
- choosing voting strategy,
- choosing stacking models,
- tuning tree depth,
- tuning number of estimators,
- selecting the final ensemble.

The test set is a final holdout.

Only after all model and ensemble decisions are frozen should the test set be evaluated.


# Phase 10 Complete

The project now demonstrates the complete progression:

```text
Single baseline
      ↓
Multiple ML models
      ↓
Optimization experiments
      ↓
Ensemble learning
      ↓
Model diversity
      ↓
Voting
      ↓
Stacking
```

This gives a strong core-ML story for placements.

---

# Phase 11 Preview — Hyperparameter Tuning & Temporal Cross-Validation

The next phase will focus on systematically improving the selected model.

Topics:

- Hyperparameter search
- Grid Search
- Randomized Search
- Bayesian-style optimization concepts
- Time-aware cross-validation
- Hyperparameter sensitivity
- Regularization tuning
- Early stopping
- Model selection under temporal drift
- Validation stability
- Final model freeze
- Final test evaluation

The key question will be:

> **How much can we improve the selected ensemble/model through principled hyperparameter optimization without leaking future information?**
